# Week 1 Day 5-7: Transformer Block + MiniGPT

**目标**: 把 Day 3-4 的 attention 拼成完整 Transformer Block,在 toy 数据上跑训练循环。

**模块组成**:
- LayerNorm (手写)
- FFN (Linear → GELU → Linear)
- Block: `x + Attention(LN(x))` → `x + FFN(LN(x))`
- 位置编码 + 多个 Block = MiniGPT

**验收点**:
- [ ] 能在白板上画完整 forward 路径
- [ ] 理解 Pre-LN (`x + Attn(LN(x))`) 比 Post-LN 更稳定
- [ ] 理解 token embedding + position embedding 为什么相加
- [ ] loss 从 ~4.5 降到 < 2.0

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

## 1. LayerNorm — 手写

`y = gamma * (x - mean) / sqrt(var + eps) + beta`

注意: mean/var 在**最后一维**算 (per-token, 不是 per-batch)

In [ ]:
class MyLayerNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(dim))
        self.beta = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x_hat = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * x_hat + self.beta

# 验证: 和 F.layer_norm 输出一致
x = torch.randn(2, 8, 64)
ln = MyLayerNorm(64)
y1 = ln(x)
y2 = F.layer_norm(x, x.shape[-1:], weight=ln.gamma, bias=ln.beta, eps=ln.eps)
diff = (y1 - y2).abs().max().item()
print(f"MyLayerNorm vs F.layer_norm diff = {diff:.2e} {'✅' if diff < 1e-5 else '❌'}")

## 2. Causal Self-Attention (复用 Day 3-4,但更紧凑)

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, D = x.shape
        qkv = self.qkv(x)                        # (B, T, 3D)
        q, k, v = qkv.chunk(3, dim=-1)           # 各 (B, T, D)
        q = q.view(B, T, self.n_heads, D // self.n_heads).transpose(1, 2)
        k = k.view(B, T, self.n_heads, D // self.n_heads).transpose(1, 2)
        v = v.view(B, T, self.n_heads, D // self.n_heads).transpose(1, 2)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.dropout(self.proj(out))

# 测试
attn = CausalSelfAttention(d_model=64, n_heads=4)
x = torch.randn(2, 8, 64)
y = attn(x)
print(f"Attention: {tuple(x.shape)} → {tuple(y.shape)} ✅")

## 3. FFN — Linear → GELU → Linear

In [ ]:
class FFN(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, 4 * d_model)
        self.fc2 = nn.Linear(4 * d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.fc2(F.gelu(self.fc1(x))))

ffn = FFN(d_model=64)
y = ffn(torch.randn(2, 8, 64))
print(f"FFN: {tuple(y.shape)} ✅")

## 4. Transformer Block — Pre-LN 残差结构

Pre-LN (GPT-2 之后主流):
```
x = x + Attention(LayerNorm(x))
x = x + FFN(LayerNorm(x))
```

Post-LN (原始 Transformer):
```
x = LayerNorm(x + Attention(x))
```

Pre-LN 在深层网络更稳定 (梯度直接走残差)

In [ ]:
class Block(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.ln1 = MyLayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = MyLayerNorm(d_model)
        self.ffn = FFN(d_model, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

block = Block(d_model=64, n_heads=4)
y = block(torch.randn(2, 8, 64))
print(f"Block: {tuple(y.shape)} ✅")

## 5. MiniGPT — 完整模型

组装: token embedding + position embedding → N 个 Block → LayerNorm → Linear

In [ ]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=4, block_size=256, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.ln_f = MyLayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        x = self.drop(x)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

# Smoke test
model = MiniGPT(vocab_size=100, d_model=64, n_heads=4, n_layers=2, block_size=32)
idx = torch.randint(0, 100, (2, 16))
targets = torch.randint(0, 100, (2, 16))
logits, loss = model(idx, targets)
n_params = sum(p.numel() for p in model.parameters())
print(f"参数量: {n_params:,}, 初始 loss: {loss.item():.3f}, 随机基线: {math.log(100):.3f}")

## 6. 训练循环 (在 tiny_shakespeare 上)

运行 `train_toy.py` 进行完整训练。这里只做模块验证。

In [ ]:
# 快速验证: 在随机数据上跑 10 步,确认训练流程通
torch.manual_seed(42)
model = MiniGPT(vocab_size=100, d_model=64, n_heads=4, n_layers=2, block_size=16)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

for step in range(10):
    x = torch.randint(0, 100, (4, 16))
    y = torch.randint(0, 100, (4, 16))
    _, loss = model(x, y)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    if step % 3 == 0:
        print(f"step {step}: loss = {loss.item():.3f}")

print("\n训练流程 ✅")

---

**交付物**:
- `MyLayerNorm` + `FFN` 完整实现
- `train_toy.py` 跑出的 `loss_curve.png` 和 `sample.txt`
- 手写笔记: Pre-LN vs Post-LN